# 02 — ExtraTrees 단일 회귀

Evidence-driven Wide HPO + anchor enqueue + 후처리 매트릭스.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/02_reg_single/et/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` ([strategy_common.md §1](../strategy_common.md))
- **HPO**: 100 trial (ET trial당 60~90초로 비싸 §8 권장값), anchor 첫 trial enqueue ([strategy.md §5.3](strategy.md), 출처: [4_output_이전자료/final/reg_only/et/best_params.json](../../4_output_이전자료/final/reg_only/et/best_params.json)), wide range ([§7.3](strategy.md))
- **손실함수**: 옵션 없음 (sklearn ExtraTrees는 MSE 고정)
- **target transform**: `'none'` 고정 (strategy_common §24 — log1p_check 검증)

## 모듈 의존성 ([strategy.md §13](strategy.md))

1. `3_modeling/modules/` 이관 완료
2. `hpo.py` — `enqueue_trials` 인자 추가
3. `postprocess.py` — `Q25/Q75` + `zero_clip_space='log'` 분기

## 1. 환경 설정 + 모듈 import

In [1]:
import os, sys

# ── Google Drive 파일 ID (Colab 사용 시. 로컬은 무시됨) ──
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip (utils/, setup.py)
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip (CSV 4개)
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip (cleaning/outlier/scaling/...)
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # ★ modules.zip Drive ID (3_modeling/modules/) — 추후 입력

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        assert GDRIVE_MODELING_ID, 'GDRIVE_MODELING_ID가 비어있음 — modules.zip Drive ID 입력 필요'
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modules.zip')
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from modules import preprocess, hpo, models   # noqa: E402
from meta_features import add_meta_features   # 2_preprocessing/meta_features.py — 2026-05-09 결정 반영

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

setup 완료


PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
Available models: ['lgbm', 'xgb', 'catboost', 'et', 'enet', 'zitboost']


## 2. 실험 설정

In [2]:
MODEL_NAME = 'et'
EXP_ID     = f'reg-{MODEL_NAME}-002'
EXP_MEMO   = 'Evidence-driven Wide + anchor enqueue (1차 OOF=0.005547)'
USER       = 'jh'

N_TRIALS = 3000
N_FOLDS  = 5
N_STARTUP_TRIALS = 50
N_JOBS   = 7
TIMEOUT_SEC      = 90 * 60 * 60  # ★ Colab 타임아웃 대비 — 초 단위, None=무제한

TARGET_TRANSFORM = 'none'  # ★ strategy_common §24 (log1p_check 검증: none=log1p 동등)
CLIP_Y_EXTREME   = True

OUT_DIR = os.path.join(OUTPUT_DIR, '02_reg_single', MODEL_NAME, EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# anchor (strategy.md §5.3, 1차 OOF=0.005547)
ET_ANCHOR = {
    'n_estimators':      583,
    'max_depth':         21,
    'min_samples_leaf':  8,
    'min_samples_split': 34,
    'max_features':      'sqrt',
}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}, N_JOBS={N_JOBS}, TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')

EXP: reg-et-002 | USER: jh
N_TRIALS=1, N_FOLDS=5, N_JOBS=7, TIMEOUT_SEC=None
TARGET_TRANSFORM=none | CLIP_Y_EXTREME=True
OUT_DIR=C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\02_reg_single\et


## 3. 데이터 로드 + target clip + transform

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# ET: target_transform 'none' 고정 (strategy_common §24 — 트리 target_transform=none 통일)
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] {TARGET_TRANSFORM} (strategy_common §24)')

[load_xs] all-NaN 행 407개 제거 → 174,573행


[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572


[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729


xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[target transform] none (strategy_common §24)


## 4. 전처리 (PP_FIXED 고정)

In [4]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# ── 메타피처 추가 (2026-05-09 결정: ET=position raw + die_xy continuous) ──
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031


[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개


    컬럼: 1031 → 926 (105개 제거)
    DataFrame: (104748, 986)



[고결측 제거] threshold=30%
  제거: 5개, 잔여: 921개


    컬럼: 926 → 921 (5개 제거)
    DataFrame: (104748, 981)



...


  [요약] 343,494 → 공간(161,870) → lot(100,428) → 전체(81,196) → 잔여(0)


[고상관 제거] threshold=0.96, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.96
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 633)

클리닝 완료: 1031 → 564 features (467개 제거)
  + indicator 컬럼: 9개 → 총 573개
  train: (104748, 633)
  val:   (34908, 633)
  test:  (34916, 633)
이상치 처리 파이프라인 시작 (method=winsorize)


[이상치 탐지] IQR × 1.5
  이상치 > 5%: 112개
  이상치 > 10%: 64개


[Winsorization] lower=0%, upper=99%
  적용 feature: 573개

이상치 처리 완료 (method=winsorize)
  train: (104748, 633)


[add_meta_features] position_mode='raw', use_die_xy=True, use_loc_x_ohe=False → position=['position'], die_xy=['die_x', 'die_y'] (feat_cols: 576)

[전처리 완료] feat_cols: 576


## 5. Optuna HPO (anchor 첫 trial enqueue + wide range)

In [5]:
study_meta_for_save = {
    'exp_id':              EXP_ID,
    'exp_memo':            EXP_MEMO,
    'user':                USER,
    'model_name':          MODEL_NAME,
    'target_transform':    TARGET_TRANSFORM,
    'clip_y_extreme':      CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':            N_TRIALS,
    'n_folds':             N_FOLDS,
    'n_jobs':              N_JOBS,
    'n_startup_trials':    N_STARTUP_TRIALS,
    'timeout_sec':         TIMEOUT_SEC,
    'seed_kfold':          SEED,
    'anchor':              ET_ANCHOR,
}

# strategy_common.md §4·§5: anchor enqueue → study 사전 생성 후 run_hpo 가 resume
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
_study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=False,
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
    pruner=MedianPruner(n_warmup_steps=10),
)
hpo.enqueue_anchor(_study, ET_ANCHOR)

res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=True,
    timeout=TIMEOUT_SEC,
    user_attrs=study_meta_for_save,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
study                 = res['study']
best_params_for_refit = res['best_params']
study_meta_for_save['hpo_best_value'] = float(res['best_value'])

# 검증 (strategy.md §15): anchor가 trial 0 인지
first_trial_params = study.trials[0].params
anchor_keys_present = {k: first_trial_params.get(k) for k in ET_ANCHOR if k in first_trial_params}
print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'[검증] trial 0 params (anchor 키만): {anchor_keys_present}')
print(f'best_params = {best_params_for_refit}')

[I 2026-05-09 16:59:28,001] A new study created in RDB with name: reg-et-002


[enqueue] anchor 첫 trial로 강제 (5 HP)


[I 2026-05-09 16:59:29,146] Using an existing study with name 'reg-et-002' instead of creating a new one.


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-05-09 17:05:06,243] Trial 0 finished with value: 0.00554805106652974 and parameters: {'n_estimators': 583, 'max_depth': 21, 'min_samples_leaf': 8, 'min_samples_split': 34, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.00554805106652974.

[HPO 완료] best OOF RMSE = 0.005548
[검증] trial 0 params (anchor 키만): {'n_estimators': 583, 'max_depth': 21, 'min_samples_leaf': 8, 'min_samples_split': 34, 'max_features': 'sqrt'}
best_params = {'n_estimators': 583, 'max_depth': 21, 'min_samples_leaf': 8, 'min_samples_split': 34, 'max_features': 'sqrt'}


## 6. Best trial 재학습 (K-fold OOF)

In [6]:
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params_for_refit,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))

y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
val_u       = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
val_rmse    = float(np.sqrt(np.mean((val_u.values - y_val_true.values)**2)))

y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
test_u      = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]
test_rmse   = float(np.sqrt(np.mean((test_u.values - y_test_true.values)**2)))

print(f'\n[Refit 완료] (original space, postprocess 이전)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

[refit fold 1/5] tr_units=20949, vl_units=5238


[refit fold 2/5] tr_units=20949, vl_units=5238


[refit fold 3/5] tr_units=20950, vl_units=5237


[refit fold 4/5] tr_units=20950, vl_units=5237


[refit fold 5/5] tr_units=20950, vl_units=5237



[Refit 완료] (original space, postprocess 이전)
  OOF  unit RMSE = 0.005542
  val  unit RMSE = 0.005758
  test unit RMSE = 0.008451


## 7. 후처리 매트릭스 + 산출물 저장

In [7]:
POSTPROCESS_CONFIG = {
    'agg_methods':      ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted', 'Q25', 'Q75'),
    'zero_clip_range':  (0.001, 0.015),
    'zero_clip_step':   0.001,
    'zero_clip_log_space': TARGET_TRANSFORM == 'log1p',
    'use_pi_threshold': False,
}

hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'{MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass

[I 2026-05-09 17:13:49,848] A new study created in memory with name: no-name-f027a81a-df4a-404f-8097-b665d00bf2ce


[Position weights / Optuna 50t] best=0.005542, w=[0.064, 0.247, 0.524, 0.165]


[Aggregation] RMSEs: {'mean': 0.005542, 'median': 0.005542, 'max': 0.005546, 'min': 0.005545, 'trimmed_mean': 0.005542, 'weighted': 0.005542, 'Q25': 0.005543, 'Q75': 0.005543}
[Aggregation] best=weighted (0.005542)


[zero_clip] best=0.0020 (0.005540)
[Postprocess] best_agg=mean, pi_th=None, zero_clip=0.002, train_rmse=0.005540, val_rmse=0.005754
  baseline_mean                  val_rmse=0.0057576507374728116
  after_agg(mean)                val_rmse=0.0057576507374728116
  after_pi_th                    val_rmse=0.0057576507374728116
  after_zero_clip                val_rmse=0.005753774506415314
  [decision] aggregation    weighted rejected (val 0.005758 <= 0.005759) -> keep mean
  [decision] zero_clip      0.0020 adopted (val 0.005758 -> 0.005754)


[save_artifacts] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\02_reg_single\et 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=tuned)
  best_params.json                      11.0 KB
  fold_models.pkl                 1,270,159.3 KB
  oof_die.csv                        5,580.5 KB
  oof_unit.csv                         900.2 KB
  optuna_jh_reg-et-002.db              112.0 KB
  test_die.csv                       1,859.8 KB
  test_unit.csv                        300.4 KB
  val_die.csv                        1,860.1 KB
  val_unit.csv                         301.1 KB
